# **1. Import Lib**

In [3]:
# Cài đặt thư viện của HuggingFace và PyTorch
!pip install transformers torch

import pandas as pd
import numpy as np
import os
import torch
import pickle
from google.colab import drive
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Kết nối với Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


# **2. Prepare Data**

In [6]:
path_project = "/content/drive/MyDrive/FakeNewsDetection_Project"
path_dataset = os.path.join(path_project, "Dataset")

# Đọc dữ liệu
df_true = pd.read_csv(os.path.join(path_dataset, "True.csv"))
df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))

# Gán nhãn
df_true['label'] = 1
df_fake['label'] = 0

# Gộp và trộn ngẫu nhiên dữ liệu
df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df[['text', 'label']].dropna()

# KHUYÊN DÙNG: Lấy một tập con để chạy thử nghiệm vì BERT trích xuất rất lâu
# Nếu muốn chạy toàn bộ, bạn hãy đổi sang: df_sub = df
df_sub = df.sample(n=2000, random_state=42).reset_index(drop=True)

print(f"Đã chuẩn bị xong dữ liệu thử nghiệm với {len(df_sub)} mẫu.")

/tmp/ipykernel_7584/3390876414.py:6: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))


Đã chuẩn bị xong dữ liệu thử nghiệm với 2000 mẫu.


# **3. BERT Mode**

In [7]:
# Tải Tokenizer và Model BERT pretrained
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Chuyển mô hình vào GPU để tăng tốc độ xử lý
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
bert_model.eval() # Chuyển sang chế độ evaluation

def get_bert_embedding(text, max_length=128):
    # Cắt văn bản và đệm (padding) về độ dài cố định, tối đa của BERT là 512
    inputs = tokenizer(
        str(text),
        return_tensors='pt',
        truncation=True,
        padding='max_length',
        max_length=max_length
    ).to(device)

    # Trích xuất vector mà không tính toán đạo hàm (để chạy nhanh hơn)
    with torch.no_grad():
        outputs = bert_model(**inputs)

    # Lấy trạng thái ẩn cuối cùng (last_hidden_state) của token [CLS]
    cls_vector = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
    return cls_vector

print("Đang tiến hành chuyển đổi văn bản thành Vector bằng BERT... (Vui lòng đợi)")
# Áp dụng hàm trích xuất cho toàn bộ tập dữ liệu thử nghiệm
X_vectors = np.array([get_bert_embedding(text) for text in df_sub['text']])
y_labels = df_sub['label'].values
print("Trích xuất vector BERT hoàn tất!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đang tiến hành chuyển đổi văn bản thành Vector bằng BERT... (Vui lòng đợi)
Trích xuất vector BERT hoàn tất!


# **4. Prepare Training Data**

In [8]:
# Import trực tiếp thư viện để tránh lỗi NameError
from sklearn.model_selection import train_test_split

# Tiến hành chia dữ liệu 80% Train và 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X_vectors,
    y_labels,
    test_size=0.2,
    random_state=42
)

print("--- SỬA LỖI THÀNH CÔNG ---")
print(f"Kích thước tập huấn luyện (Train): {X_train.shape}")
print(f"Kích thước tập kiểm thử (Test): {X_test.shape}")

--- SỬA LỖI THÀNH CÔNG ---
Kích thước tập huấn luyện (Train): (1600, 768)
Kích thước tập kiểm thử (Test): (400, 768)


# **5. Training with Naive Bayes**

In [10]:
# Khởi tạo thuật toán Naive Bayes dạng Gaussian (phù hợp với vector số thực liên tục từ BERT)
nb_model = GaussianNB()

# Huấn luyện mô hình
nb_model.fit(X_train, y_train)

# Dự đoán dữ liệu kiểm thử
y_pred = nb_model.predict(X_test)

# Tính toán các chỉ số APRF
metrics = {
    'acc': accuracy_score(y_test, y_pred),
    'pre': precision_score(y_test, y_pred),
    'rec': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print("\n" + "="*40)
print("KẾT QUẢ THỰC NGHIỆM: BERT + NAIVE BAYES")
print("="*40)
print(f"1. Accuracy  (A): {metrics['acc']:.4f}")
print(f"2. Precision (P): {metrics['pre']:.4f}")
print(f"3. Recall    (R): {metrics['rec']:.4f}")
print(f"4. F1-Score  (F): {metrics['f1']:.4f}")
print("="*40)
# ==========================================================
# CẤU HÌNH BỔ SUNG: LƯU MÔ HÌNH VÀO DRIVE CHO ỨNG DỤNG DEMO
# ==========================================================
# Khai báo lại đường dẫn trực tiếp tại đây để tránh lỗi NameError
path_project = "/content/drive/MyDrive/FakeNewsDetection_Project"
path_models = os.path.join(path_project, "Models")

# Kiểm tra và tự động tạo thư mục Models nếu Drive chưa có
if not os.path.exists(path_models):
    os.makedirs(path_models)

file_save_path = os.path.join(path_models, "bert_nb_classifier.pkl")
print(f"\nĐang đóng gói và lưu mô hình vào Google Drive...")

with open(file_save_path, 'wb') as f:
    pickle.dump(nb_model, f)

print(f"✅ THÀNH CÔNG: Mô hình đã được lưu tại: {file_save_path}")
# ==========================================================


KẾT QUẢ THỰC NGHIỆM: BERT + NAIVE BAYES
1. Accuracy  (A): 0.9425
2. Precision (P): 0.9637
3. Recall    (R): 0.9208
4. F1-Score  (F): 0.9418

Đang đóng gói và lưu mô hình vào Google Drive...
✅ THÀNH CÔNG: Mô hình đã được lưu tại: /content/drive/MyDrive/FakeNewsDetection_Project/Models/bert_nb_classifier.pkl


# **6. Test Sentence Real or Fake**

In [11]:
def predict_news_bert_nb(sentence):
    # 1. Trích xuất vector từ câu mới bằng BERT
    vector = get_bert_embedding(sentence).reshape(1, -1)
    # 2. Đưa vào mô hình Naive Bayes dự đoán
    prediction = nb_model.predict(vector)[0]
    return "TIN THẬT (REAL)" if prediction == 1 else "TIN GIẢ (FAKE)"

# Thử nghiệm thực tế với 1 câu
sample_sentence = "The international community agreed to implement new carbon emission standards during the summit."
print(f"\nCâu test: '{sample_sentence}'")
print(f"Mô hình BERT + NB dự đoán: {predict_news_bert_nb(sample_sentence)}")


Câu test: 'The international community agreed to implement new carbon emission standards during the summit.'
Mô hình BERT + NB dự đoán: TIN THẬT (REAL)
